# SUDS Scholar — Agentic Document Retrieval via Summarization Trees

Implements a tree-structured RAG system that mimics human browsing of a document library.

**Architecture**
1. Parse every document in `folders/` (PDF, DOCX, CSV, XLSX)
2. Chunk text → summarize chunks → summarize pages → summarize documents → summarize folders (bottom-up)
3. At query time, an LLM agent traverses the tree top-down to find relevant chunks
4. Compare against BM25 and vector-search RAG baselines

**Setup checklist (run once before the notebook)**
```bash
# 1. Start SSH tunnel to remote Ollama (leave this terminal open)
ssh -N -L 11528:172.17.0.1:11434 asharma@ollama.res.oicr.on.ca

# 2. Install Python dependencies
pip install -r requirements.txt

# 3. Verify the tunnel works
curl http://localhost:11528/api/tags

# 4. Pull the required models (if not already on the server)
#    Run these in a SEPARATE terminal with OLLAMA_HOST set:
export OLLAMA_HOST=localhost:11528
ollama pull gemma3:270m
ollama pull nomic-embed-text

# 5. Populate folders/ with your documents, then run this notebook top-to-bottom
```

## 1 · Imports & package check

In [1]:
%pip install ollama pymupdf python-docx pandas openpyxl rank-bm25 tqdm numpy

Note: you may need to restart the kernel to use updated packages.


In [2]:
import importlib, sys, os, json, hashlib, re, time, textwrap
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Any, Optional, Tuple

required = {
    "ollama": "ollama",
    "fitz": "pymupdf",
    "docx": "python-docx",
    "pandas": "pandas",
    "openpyxl": "openpyxl",
    "rank_bm25": "rank-bm25",
    "tqdm": "tqdm",
    "numpy": "numpy",
}
missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    print(f"Missing — run:  pip install {' '.join(missing)}")
    raise SystemExit(1)

import ollama
import fitz                          # PyMuPDF
import docx as _docx
import pandas as pd
import numpy as np
from rank_bm25 import BM25Okapi
from tqdm import tqdm                # plain text bar; no ipywidgets needed

print("All packages OK")

All packages OK


## 2 · Configuration — edit this cell

In [3]:
# ---------------------------------------------------------------------------
# Ollama
# ---------------------------------------------------------------------------
OLLAMA_URL   = "http://localhost:11528"   # SSH tunnel endpoint

# Run `ollama list` (with OLLAMA_HOST=localhost:11528) to see what is available.
# Small model on purpose: this is a proof-of-concept, so we favour speed.
SUMMARY_MODEL = "gemma3:270m"       # used to generate all summaries
AGENT_MODEL   = "gemma3:270m"       # used for traversal decisions
CHAT_MODEL    = "gemma3:270m"       # used for final answer synthesis
EMBED_MODEL   = "nomic-embed-text"  # used for vector-search baseline

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
DOCS_ROOT  = Path("folders")        # root of document library
CACHE_DIR  = Path("tree_cache")
TREE_FILE  = CACHE_DIR / "summary_tree.json"
INDEX_FILE = CACHE_DIR / "rag_index.json"
NODE_CACHE_DIR = CACHE_DIR / "nodes"   # per-node cache for crash-resumable builds

# ---------------------------------------------------------------------------
# Retrieval
# ---------------------------------------------------------------------------
MAX_CHUNKS_RET  = 6      # max leaf paragraphs returned per tree-agent query
# Deliberately NOT configured here:
#   * No chunk-size / overlap knobs — documents are split STRUCTURALLY
#     (paragraphs -> sections/pages -> document), never by word count.
#   * No summary-token cap — summaries are kept short via the prompt
#     ("at most 200 words"); inputs and outputs are not truncated.
# All generation runs at temperature 0 (deterministic / reproducible).

# Set True to rebuild even when a cached tree exists
FORCE_REBUILD = True

CACHE_DIR.mkdir(exist_ok=True)
NODE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Docs root : {DOCS_ROOT.resolve()}")
print(f"Tree file : {TREE_FILE}")
print(f"Node cache: {NODE_CACHE_DIR}/  (per-node, crash-resumable)")
print(f"Models    : summarise={SUMMARY_MODEL}  agent={AGENT_MODEL}  embed={EMBED_MODEL}")

Docs root : /Users/asharma/Desktop/Project Algorithm/folders
Tree file : tree_cache/summary_tree.json
Node cache: tree_cache/nodes/  (per-node, crash-resumable)
Models    : summarise=gemma3:270m  agent=gemma3:270m  embed=nomic-embed-text


## 3 · Ollama client & connection test

In [4]:
# A request timeout means a stuck call raises instead of hanging forever.
# Bump OLLAMA_TIMEOUT if your first call must cold-load a large model.
OLLAMA_TIMEOUT = 120   # seconds, per request
client = ollama.Client(host=OLLAMA_URL, timeout=OLLAMA_TIMEOUT)


def _model_names(list_resp) -> List[str]:
    """Extract model names robustly across ollama-python versions.

    Newer clients return objects whose name field is `.model` (not `['name']`),
    which is why `m['name']` was raising KeyError: 'name'.
    """
    raw = list_resp.get("models", []) if hasattr(list_resp, "get") else getattr(list_resp, "models", [])
    out = []
    for m in raw:
        name = getattr(m, "model", None) or getattr(m, "name", None)
        if name is None and isinstance(m, dict):
            name = m.get("model") or m.get("name")
        if name:
            out.append(name)
    return out


try:
    models = client.list()
    names  = _model_names(models)
    print("Connected to Ollama. Available models:")
    for n in names:
        print(f"  {n}")
    for required_model in [SUMMARY_MODEL, EMBED_MODEL]:
        if not any(required_model in n for n in names):
            print(f"\nWARNING: '{required_model}' not found. Pull it with:")
            print(f"  OLLAMA_HOST=localhost:11528 ollama pull {required_model}")
except Exception as e:
    print(f"Cannot reach Ollama at {OLLAMA_URL}: {type(e).__name__}: {e}")
    print("Make sure the SSH tunnel is running:")
    print("  ssh -N -L 11528:172.17.0.1:11434 asharma@ollama.res.oicr.on.ca")

Connected to Ollama. Available models:
  nomic-embed-text:latest
  gemma3:270m
  llama3.1:latest
  llama3.1:8b
  medgemma:27b
  gpt-oss:20b
  gemma3:27b
  codellama:latest
  llama3.1:70b
  mistral:latest
  gpt-oss:120b
  llama3:70b-instruct
  mistral-small3.1:latest


In [5]:
# Latency probe — confirms chat() actually works and shows per-call time.
# If this is fast but the build "hangs", the build is just slow (many chunks),
# not stuck. If this hangs/times out, the problem is inference or the tunnel.
_t0 = time.time()
try:
    _r = client.chat(
        model=SUMMARY_MODEL,
        messages=[{"role": "user", "content": "Reply with the single word: ok"}],
        options={"num_predict": 5},
    )
    print(f"chat() OK in {time.time() - _t0:.1f}s  →  {_r['message']['content'].strip()!r}")
except Exception as e:
    print(f"chat() FAILED after {time.time() - _t0:.1f}s: {type(e).__name__}: {e}")

chat() OK in 2.6s  →  'Okay'


## 4 · Document loaders
Parse PDF, DOCX, CSV, and XLSX into a list of `{page, text}` dicts per document.

In [6]:
def _clean(text: str) -> str:
    """Normalise whitespace within a paragraph unit."""
    return re.sub(r"\s+", " ", text).strip()


def load_pdf_structured(path: Path) -> List[Dict]:
    """Structure-aware PDF loader.

    Uses PyMuPDF layout 'blocks' (each block ~= a paragraph) grouped by page.
    Returns one entry per page: {title, page, paragraphs}. No word-count split.
    """
    pages = []
    try:
        doc = fitz.open(str(path))
        for i, page in enumerate(doc):
            blocks = page.get_text("blocks")  # (x0, y0, x1, y1, text, block_no, block_type)
            paras = []
            for b in sorted(blocks, key=lambda blk: (round(blk[1]), blk[0])):
                if len(b) >= 7 and b[6] != 0:
                    continue  # skip non-text (e.g. image) blocks
                txt = _clean(b[4]) if len(b) > 4 and isinstance(b[4], str) else ""
                if txt:
                    paras.append(txt)
            if paras:
                pages.append({"title": f"Page {i + 1}", "page": i + 1, "paragraphs": paras})
        doc.close()
    except Exception as e:
        print(f"  PDF error {path.name}: {e}")
    return pages


def load_docx_structured(path: Path) -> List[Dict]:
    """Structure-aware DOCX loader.

    Walks paragraphs in order; a heading-styled paragraph (Heading 1/2/..., Title)
    starts a new section and the paragraphs beneath it are that section's content.
    Returns one entry per section: {title, paragraphs}. No word-count split.
    """
    sections: List[Dict] = []
    try:
        document = _docx.Document(str(path))
        state = {"title": None, "paras": []}

        def flush():
            if state["paras"]:
                sections.append({"title": state["title"] or "(body)",
                                 "paragraphs": list(state["paras"])})

        for para in document.paragraphs:
            t = _clean(para.text)
            if not t:
                continue
            style = ((para.style.name if para.style else "") or "").lower()
            if style.startswith("heading") or style.startswith("title"):
                flush()
                state["title"] = t
                state["paras"] = []
            else:
                state["paras"].append(t)
        flush()

        # Fall back to one section if the document has no headings at all.
        if not sections:
            allp = [_clean(p.text) for p in document.paragraphs if _clean(p.text)]
            if allp:
                sections = [{"title": path.stem, "paragraphs": allp}]
    except Exception as e:
        print(f"  DOCX error {path.name}: {e}")
    return sections


def load_structured(path: Path) -> List[Dict]:
    """Dispatch to the structure-aware loader for the file type.

    Returns a list of sections, each a dict:
      'title'      — section/page label
      'paragraphs' — list of paragraph strings (the leaf units)
      'page'       — page number (PDF only; absent for DOCX)
    Only .pdf and .docx are supported; everything else is skipped.
    """
    ext = path.suffix.lower()
    if ext == ".pdf":
        return load_pdf_structured(path)
    elif ext == ".docx":
        return load_docx_structured(path)
    return []


# Spreadsheets (.xlsx/.xls/.csv) and images (.bmp) are intentionally skipped.
SUPPORTED_EXTENSIONS = {".pdf", ".docx"}
print("Structure-aware document loaders ready (pdf + docx only)")

Structure-aware document loaders ready (pdf + docx only)


## 6 · Tree data structure

In [7]:
@dataclass
class TreeNode:
    """
    A node in the summarisation tree.
    node_type: 'root' | 'folder' | 'document' | 'chunk'
    """
    node_id:   str
    node_type: str
    name:      str
    path:      str          # filesystem path (for folders/documents) or '' for chunks
    summary:   str          # LLM-generated summary
    content:   str = ""     # raw text (only for chunk nodes)
    children:  List["TreeNode"] = field(default_factory=list)
    metadata:  Dict[str, Any]   = field(default_factory=dict)

    # ---- serialisation ----
    def to_dict(self) -> Dict:
        return {
            "node_id":   self.node_id,
            "node_type": self.node_type,
            "name":      self.name,
            "path":      self.path,
            "summary":   self.summary,
            "content":   self.content,
            "children":  [c.to_dict() for c in self.children],
            "metadata":  self.metadata,
        }

    @classmethod
    def from_dict(cls, d: Dict) -> "TreeNode":
        node = cls(
            node_id   = d["node_id"],
            node_type = d["node_type"],
            name      = d["name"],
            path      = d["path"],
            summary   = d["summary"],
            content   = d.get("content", ""),
            metadata  = d.get("metadata", {}),
        )
        node.children = [cls.from_dict(c) for c in d.get("children", [])]
        return node

    # ---- helpers ----
    def is_leaf(self) -> bool:
        return self.node_type == "chunk"

    def count_nodes(self) -> int:
        return 1 + sum(c.count_nodes() for c in self.children)

    def count_leaves(self) -> int:
        if self.is_leaf():
            return 1
        return sum(c.count_leaves() for c in self.children)


def _make_id(path: str, extra: str = "") -> str:
    return hashlib.md5(f"{path}|{extra}".encode()).hexdigest()[:12]

print("TreeNode class ready")

TreeNode class ready


## 7 · LLM helper functions

In [8]:
def _llm_summarise(text: str, context_hint: str = "", max_retries: int = 3) -> str:
    """Ask the LLM for a concise summary suitable for navigation."""
    if not text.strip():
        return "(empty)"
    hint = f" The content comes from: {context_hint}." if context_hint else ""
    prompt = (
        f"Summarise the following text.{hint} "
        "Focus on the key topics, concepts, and specific information present — "
        "your summary will be used to help someone decide whether to read this section. "
        "Be specific (mention names, numbers, procedures, entities). "
        "Keep your summary to AT MOST 200 words. "
        "Do not add commentary or caveats. Respond with ONLY the summary.\n\n"
        f"{text}"
    )
    for attempt in range(max_retries):
        try:
            resp = client.chat(
                model=SUMMARY_MODEL,
                messages=[{"role": "user", "content": prompt}],
                options={"temperature": 0},
            )
            return resp["message"]["content"].strip()
        except Exception as e:
            if attempt == max_retries - 1:
                return f"(summary unavailable: {e})"
            time.sleep(2 ** attempt)


def _llm_combine_summaries(summaries: List[str], label: str, max_retries: int = 3) -> str:
    """Summarise a list of child summaries into one parent summary."""
    if not summaries:
        return "(no content)"
    joined = "\n".join(f"- {s}" for s in summaries)
    prompt = (
        f"You are summarising a section, document, or folder called '{label}'. "
        "Below are summaries of its contents. Write a single summary covering the "
        "overall scope and key topics. Be specific. "
        "Keep your summary to AT MOST 200 words. "
        "Respond with ONLY the summary.\n\n"
        f"{joined}"
    )
    for attempt in range(max_retries):
        try:
            resp = client.chat(
                model=SUMMARY_MODEL,
                messages=[{"role": "user", "content": prompt}],
                options={"temperature": 0},
            )
            return resp["message"]["content"].strip()
        except Exception as e:
            if attempt == max_retries - 1:
                return f"(summary unavailable: {e})"
            time.sleep(2 ** attempt)


def _llm_embed(text: str) -> Optional[List[float]]:
    """Get a vector embedding for text via Ollama."""
    try:
        resp = client.embeddings(model=EMBED_MODEL, prompt=text)
        return resp["embedding"]
    except Exception:
        return None


print("LLM helpers ready")

LLM helpers ready


## 8 · Summarisation tree builder

**Bottom-up construction**
```
chunks  ──summarise──►  document node
document nodes ──summarise──►  folder node
folder nodes  ──summarise──►  parent folder node
                                    ...
                               root node
```

In [9]:
def build_document_node(file_path: Path, verbose: bool = False) -> Optional[TreeNode]:
    """Build a document subtree from the file's natural structure (no word-count chunking).

      * DOCX -> paragraphs grouped under their headings (sections)
      * PDF  -> paragraphs (layout blocks) grouped by page
    Tree shape:  document -> section/page -> paragraph (leaf).
    Each paragraph is summarised, each section summarises its paragraphs, and the
    document summarises its sections. Per-node caching persists every paragraph as
    soon as it is summarised (crash-resumable).
    """
    doc_id = _make_id(str(file_path))

    # Fast path: this document was fully built in a previous run.
    cached_doc = _load_cached_node(doc_id)
    if cached_doc is not None and cached_doc.node_type == "document":
        if verbose:
            print(f"    {file_path.name}  ->  loaded from cache")
        return cached_doc

    if verbose:
        print(f"    Parsing {file_path.name} (structure-aware) ...", flush=True)

    sections_data = load_structured(file_path)
    if not sections_data:
        if verbose:
            print("      no content, skipped")
        return None

    # Flatten paragraphs across sections for one progress bar, keeping the
    # section index so we can regroup afterwards.
    tasks: List[Tuple[int, int, str, Optional[int], str]] = []
    for si, sec in enumerate(sections_data):
        page = sec.get("page")
        for pi, para in enumerate(sec["paragraphs"]):
            tasks.append((si, pi, sec["title"], page, para))

    if verbose:
        print(f"      {len(sections_data)} section(s), {len(tasks)} paragraph(s)", flush=True)

    para_by_section: Dict[int, List[TreeNode]] = {}
    fresh = hits = 0
    for si, pi, title, page, para in tqdm(
        tasks, desc=f"  {file_path.name[:28]}", leave=False, disable=not verbose
    ):
        cid = _make_id(str(file_path), f"s{si}p{pi}")
        cached_p = _load_cached_node(cid)
        if cached_p is not None and cached_p.node_type == "chunk":
            para_by_section.setdefault(si, []).append(cached_p)
            hits += 1
            continue

        meta = {
            "source_file":     str(file_path),
            "section":         title,
            "paragraph_index": pi,
            "file_type":       file_path.suffix.lower(),
        }
        if page is not None:
            meta["page"] = page

        summary = _llm_summarise(para, context_hint=f"{file_path.name}, {title}")
        pnode = TreeNode(
            node_id   = cid,
            node_type = "chunk",
            name      = f"{file_path.name} - {title} - para {pi + 1}",
            path      = "",
            summary   = summary,
            content   = para,
            metadata  = meta,
        )
        _save_cached_node(pnode)   # persist immediately (crash-resumable)
        para_by_section.setdefault(si, []).append(pnode)
        fresh += 1

    # One section node per section that has paragraphs.
    section_nodes: List[TreeNode] = []
    for si, sec in enumerate(sections_data):
        pnodes = para_by_section.get(si, [])
        if not pnodes:
            continue
        sec_meta = {
            "source_file": str(file_path),
            "section":     sec["title"],
            "file_type":   file_path.suffix.lower(),
        }
        if sec.get("page") is not None:
            sec_meta["page"] = sec["page"]
        sec_summary = _llm_combine_summaries([p.summary for p in pnodes], label=sec["title"])
        snode = TreeNode(
            node_id   = _make_id(str(file_path), f"s{si}"),
            node_type = "section",
            name      = f"{file_path.name} - {sec['title']}",
            path      = "",
            summary   = sec_summary,
            children  = pnodes,
            metadata  = sec_meta,
        )
        _save_cached_node(snode)
        section_nodes.append(snode)

    if not section_nodes:
        return None

    doc_summary = _llm_combine_summaries([s.summary for s in section_nodes], label=file_path.name)
    if verbose:
        print(f"      done: {len(section_nodes)} section(s)  (fresh paras={fresh}, cached={hits})")

    doc_node = TreeNode(
        node_id   = doc_id,
        node_type = "document",
        name      = file_path.name,
        path      = str(file_path),
        summary   = doc_summary,
        children  = section_nodes,
        metadata  = {"file_type": file_path.suffix.lower(), "num_sections": len(section_nodes)},
    )
    _save_cached_node(doc_node)
    return doc_node


def build_folder_node(folder_path: Path, is_root: bool = False, verbose: bool = True) -> Optional[TreeNode]:
    """Recursively build a folder node from all documents and sub-folders."""
    folder_id = _make_id(str(folder_path))

    # Fast path: this whole folder was already fully built and cached.
    cached_folder = _load_cached_node(folder_id)
    if cached_folder is not None and cached_folder.node_type in ("folder", "root"):
        if verbose:
            print(f"  Folder: {folder_path.name or folder_path}  →  loaded from cache "
                  f"({cached_folder.count_leaves()} leaf chunks)")
        # Make sure the node_type matches whether we are at root now or not.
        cached_folder.node_type = "root" if is_root else "folder"
        return cached_folder

    if verbose:
        print(f"  Folder: {folder_path.name or folder_path}")

    children: List[TreeNode] = []

    # Sort entries: folders first, then files
    entries = sorted(folder_path.iterdir(), key=lambda p: (p.is_file(), p.name.lower()))

    for entry in entries:
        if entry.name.startswith("."):
            continue  # skip hidden files
        if entry.is_dir():
            child = build_folder_node(entry, verbose=verbose)
            if child:
                children.append(child)
        elif entry.is_file() and entry.suffix.lower() in SUPPORTED_EXTENSIONS:
            child = build_document_node(entry, verbose=verbose)
            if child:
                children.append(child)

    if not children:
        if verbose:
            print(f"    (empty or no supported files)")
        return None

    folder_summary = _llm_combine_summaries(
        [c.summary for c in children],
        label=folder_path.name or "root"
    )

    node_type = "root" if is_root else "folder"
    folder_node = TreeNode(
        node_id   = folder_id,
        node_type = node_type,
        name      = folder_path.name or str(folder_path),
        path      = str(folder_path),
        summary   = folder_summary,
        children  = children,
        metadata  = {"num_children": len(children)},
    )
    # Persist this folder once its subtree is fully built.
    _save_cached_node(folder_node)
    return folder_node


def build_tree(root_path: Path = DOCS_ROOT, verbose: bool = True) -> Optional[TreeNode]:
    """Build the full summarisation tree from the document library root.

    Crash-resumable: per-node cache files in `NODE_CACHE_DIR` mean that
    re-running this function picks up exactly where the previous run stopped
    (granularity = one paragraph).
    """
    print(f"Building summarisation tree from: {root_path.resolve()}")
    print(f"Per-node cache: {NODE_CACHE_DIR}  (delete to force a full rebuild)")
    t0 = time.time()
    tree = build_folder_node(root_path, is_root=True, verbose=verbose)
    elapsed = time.time() - t0
    if tree:
        print(f"\nTree built in {elapsed:.1f}s")
        print(f"  Total nodes : {tree.count_nodes()}")
        print(f"  Leaf chunks : {tree.count_leaves()}")
    else:
        print("Tree is empty — make sure folders/ contains documents.")
    return tree


print("Tree builder functions ready (structure-aware + per-node caching)")

Tree builder functions ready (structure-aware + per-node caching)


## 8.5 · Per-node cache (crash-resumable build)

Every chunk/document/folder summary is written to its own JSON file in `tree_cache/nodes/` **immediately** after the LLM produces it. On restart, each node first looks for a cached result and skips the LLM call if one exists, so a crash mid-build only loses the chunk that was in flight.

In [10]:
def _node_cache_path(node_id: str) -> Path:
    return NODE_CACHE_DIR / f"{node_id}.json"


def _load_cached_node(node_id: str) -> Optional["TreeNode"]:
    """Load a fully-cached TreeNode (with its subtree) if present, else None."""
    p = _node_cache_path(node_id)
    if not p.exists():
        return None
    try:
        with open(p, encoding="utf-8") as f:
            return TreeNode.from_dict(json.load(f))
    except Exception as e:
        print(f"  (cache read failed for {node_id}: {e} — rebuilding)")
        return None


def _save_cached_node(node: "TreeNode") -> None:
    """Atomically persist a TreeNode (with its subtree) to its own cache file."""
    p = _node_cache_path(node.node_id)
    tmp = p.with_suffix(".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(node.to_dict(), f, ensure_ascii=False)
    tmp.replace(p)   # atomic on POSIX — survives kill -9 mid-write


def clear_node_cache() -> int:
    """Delete every per-node cache file. Returns count removed."""
    n = 0
    for p in NODE_CACHE_DIR.glob("*.json"):
        p.unlink()
        n += 1
    print(f"Cleared {n} cached node(s) from {NODE_CACHE_DIR}")
    return n


def cache_status() -> None:
    """Report how many nodes of each type are cached."""
    counts = {"chunk": 0, "document": 0, "folder": 0, "root": 0, "other": 0}
    total_bytes = 0
    for p in NODE_CACHE_DIR.glob("*.json"):
        total_bytes += p.stat().st_size
        try:
            with open(p, encoding="utf-8") as f:
                d = json.load(f)
            counts[d.get("node_type", "other")] = counts.get(d.get("node_type", "other"), 0) + 1
        except Exception:
            counts["other"] += 1
    print(f"Per-node cache @ {NODE_CACHE_DIR}:")
    for k, v in counts.items():
        if v:
            print(f"  {k:<10s}: {v}")
    print(f"  total size: {total_bytes / 1024:.1f} KB")


print("Per-node cache helpers ready")

Per-node cache helpers ready


## 9 · Save / load tree

In [11]:
def save_tree(tree: TreeNode, path: Path = TREE_FILE) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(tree.to_dict(), f, ensure_ascii=False, indent=2)
    print(f"Tree saved → {path}  ({path.stat().st_size / 1024:.1f} KB)")


def load_tree(path: Path = TREE_FILE) -> Optional[TreeNode]:
    if not path.exists():
        return None
    with open(path, encoding="utf-8") as f:
        return TreeNode.from_dict(json.load(f))


print("Save/load helpers ready")

Save/load helpers ready


## 10 · Agentic tree traversal

Given a query, the agent starts at the root and at each node asks the LLM  
"Which of these children are relevant?" before descending — mimicking how a  
human navigates a filing cabinet.

In [12]:
def _agent_select_children(
    query: str,
    node: TreeNode,
    indent: str = "",
    verbose: bool = True,
) -> List[int]:
    """Ask the LLM which children of `node` are relevant to `query`.
    Returns a list of 0-based indices.
    """
    children_text = "\n".join(
        f"{i+1}. [{c.node_type.upper()}] {c.name}\n   {c.summary}"
        for i, c in enumerate(node.children)
    )
    prompt = (
        f"You are navigating a document library to answer a query.\n\n"
        f"Query: \"{query}\"\n\n"
        f"Current location: {node.name} ({node.node_type})\n"
        f"Location summary: {node.summary}\n\n"
        f"Available sub-sections:\n{children_text}\n\n"
        "Which sub-sections likely contain relevant information?\n"
        "Rules:\n"
        "- Respond with ONLY a comma-separated list of numbers (e.g. \"1, 3\")\n"
        "- Respond with NONE if nothing seems relevant\n"
        "- Be inclusive rather than exclusive — better to look at more sections"
    )
    resp = client.chat(
        model=AGENT_MODEL,
        messages=[{"role": "user", "content": prompt}],
        options={"num_predict": 60, "temperature": 0.0},
    )
    answer = resp["message"]["content"].strip()

    if verbose:
        print(f"{indent}[{node.node_type.upper()}] {node.name}  →  agent chose: {answer}")

    if answer.upper() == "NONE":
        return []

    indices = []
    for token in re.split(r"[,\s]+", answer):
        token = token.strip().rstrip(".")
        if token.isdigit():
            idx = int(token) - 1
            if 0 <= idx < len(node.children):
                indices.append(idx)
    return indices


def traverse(
    query: str,
    node: TreeNode,
    collected: List[TreeNode],
    max_chunks: int = MAX_CHUNKS_RET,
    indent: str = "",
    verbose: bool = True,
) -> None:
    """Recursively traverse the tree, appending relevant chunks to `collected`."""
    if len(collected) >= max_chunks:
        return
    if node.is_leaf():
        collected.append(node)
        return
    if not node.children:
        return

    chosen_indices = _agent_select_children(query, node, indent=indent, verbose=verbose)

    for idx in chosen_indices:
        if len(collected) >= max_chunks:
            break
        traverse(
            query, node.children[idx], collected,
            max_chunks=max_chunks,
            indent=indent + "  ",
            verbose=verbose,
        )


def synthesise_answer(query: str, chunks: List[TreeNode]) -> str:
    """Ask the LLM to answer the query using the retrieved chunks."""
    if not chunks:
        return "No relevant documents found in the library for this query."

    context_parts = []
    for i, chunk in enumerate(chunks):
        source = chunk.metadata.get("source_file", "unknown")
        page   = chunk.metadata.get("page", "?")
        context_parts.append(
            f"[Source {i+1}: {Path(source).name}, page {page}]\n{chunk.content}"
        )
    context = "\n\n".join(context_parts)

    prompt = (
        f"Answer the following question using ONLY the provided document excerpts.\n\n"
        f"Question: {query}\n\n"
        f"Document excerpts:\n{context[:6000]}\n\n"
        "Provide a comprehensive, well-structured answer. "
        "Cite sources by their [Source N] labels. "
        "If the answer is not found in the excerpts, say so explicitly."
    )
    resp = client.chat(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0},
    )
    return resp["message"]["content"].strip()


def query_tree_agent(
    query: str,
    tree: TreeNode,
    max_chunks: int = MAX_CHUNKS_RET,
    verbose: bool = True,
) -> Dict[str, Any]:
    """Full pipeline: traverse tree → retrieve chunks → synthesise answer."""
    print(f"\n{'='*60}")
    print(f"Query : {query}")
    print(f"{'='*60}")
    print("\nTraversing summarisation tree...\n")

    collected: List[TreeNode] = []
    t0 = time.time()
    traverse(query, tree, collected, max_chunks=max_chunks, verbose=verbose)
    traverse_time = time.time() - t0

    print(f"\nRetrieved {len(collected)} chunk(s) in {traverse_time:.1f}s")
    for c in collected:
        src  = Path(c.metadata.get("source_file", "?")).name
        page = c.metadata.get("page", "?")
        print(f"  • {src}  (page {page})  — {c.summary[:80]}...")

    print("\nSynthesising answer...")
    t1 = time.time()
    answer = synthesise_answer(query, collected)
    synth_time = time.time() - t1

    print(f"\nAnswer (synthesised in {synth_time:.1f}s):")
    print("-" * 60)
    print(answer)
    print("-" * 60)

    return {
        "query":         query,
        "answer":        answer,
        "chunks":        collected,
        "traverse_time": traverse_time,
        "synth_time":    synth_time,
    }


print("Tree agent ready")

Tree agent ready


## 11 · RAG baseline (BM25 + vector search)

Flat retrieval over all chunks — no awareness of folder structure.

In [13]:
def collect_all_chunks(node: TreeNode) -> List[TreeNode]:
    """Flatten the tree into a list of all chunk nodes."""
    if node.is_leaf():
        return [node]
    chunks = []
    for child in node.children:
        chunks.extend(collect_all_chunks(child))
    return chunks


class RAGIndex:
    """BM25 + optional vector index over the full chunk corpus."""

    def __init__(self, chunks: List[TreeNode]):
        self.chunks = chunks
        texts = [c.content for c in chunks]

        # BM25
        tokenised = [t.lower().split() for t in texts]
        self.bm25 = BM25Okapi(tokenised)

        # Vector (optional — can fail gracefully if model unavailable)
        self.embeddings: Optional[np.ndarray] = None

    def build_vector_index(self, verbose: bool = True) -> None:
        """Embed all chunks with Ollama. Can take a few minutes for large libraries."""
        print(f"Embedding {len(self.chunks)} chunks with '{EMBED_MODEL}'...")
        vecs = []
        for chunk in tqdm(self.chunks, desc="Embedding"):
            vec = _llm_embed(chunk.content)
            if vec is None:
                vec = [0.0] * 768  # fallback zero vector
            vecs.append(vec)
        self.embeddings = np.array(vecs, dtype=np.float32)
        norms = np.linalg.norm(self.embeddings, axis=1, keepdims=True)
        norms = np.where(norms == 0, 1, norms)
        self.embeddings /= norms  # L2-normalise for fast cosine via dot product
        print("Vector index built.")

    def retrieve_bm25(self, query: str, top_k: int = MAX_CHUNKS_RET) -> List[Tuple[float, TreeNode]]:
        scores = self.bm25.get_scores(query.lower().split())
        ranked = np.argsort(scores)[::-1][:top_k]
        return [(float(scores[i]), self.chunks[i]) for i in ranked if scores[i] > 0]

    def retrieve_vector(self, query: str, top_k: int = MAX_CHUNKS_RET) -> List[Tuple[float, TreeNode]]:
        if self.embeddings is None:
            raise RuntimeError("Call build_vector_index() first.")
        q_vec = _llm_embed(query)
        if q_vec is None:
            return []
        q_arr = np.array(q_vec, dtype=np.float32)
        q_arr /= max(np.linalg.norm(q_arr), 1e-9)
        sims = self.embeddings @ q_arr
        ranked = np.argsort(sims)[::-1][:top_k]
        return [(float(sims[i]), self.chunks[i]) for i in ranked]


def query_rag(
    query: str,
    rag_index: RAGIndex,
    method: str = "bm25",   # 'bm25' or 'vector'
    top_k: int = MAX_CHUNKS_RET,
) -> Dict[str, Any]:
    """Retrieve chunks via BM25 or vector search and synthesise an answer."""
    print(f"\n{'='*60}")
    print(f"Query [{method.upper()}]: {query}")
    print("="*60)

    t0 = time.time()
    if method == "bm25":
        results = rag_index.retrieve_bm25(query, top_k=top_k)
    else:
        results = rag_index.retrieve_vector(query, top_k=top_k)
    retrieve_time = time.time() - t0

    chunks = [chunk for _, chunk in results]
    print(f"Retrieved {len(chunks)} chunk(s) in {retrieve_time:.3f}s")
    for score, chunk in results:
        src  = Path(chunk.metadata.get("source_file", "?")).name
        page = chunk.metadata.get("page", "?")
        print(f"  • score={score:.3f}  {src}  p{page}")

    print("\nSynthesising answer...")
    t1 = time.time()
    answer = synthesise_answer(query, chunks)
    synth_time = time.time() - t1

    print(f"\nAnswer:")
    print("-" * 60)
    print(answer)
    print("-" * 60)

    return {
        "query":         query,
        "answer":        answer,
        "chunks":        chunks,
        "retrieve_time": retrieve_time,
        "synth_time":    synth_time,
    }


print("RAG index class ready")

RAG index class ready


## 12 · Build (or load) the summarisation tree

Set `FORCE_REBUILD = True` in the config cell to rebuild from scratch.

In [ ]:
tree: Optional[TreeNode] = None

# If the user explicitly asked for a clean rebuild, wipe the per-node cache too.
if FORCE_REBUILD:
    clear_node_cache()

if not FORCE_REBUILD and TREE_FILE.exists():
    print(f"Loading cached tree from {TREE_FILE} ...")
    tree = load_tree(TREE_FILE)
    if tree:
        print(f"Loaded — {tree.count_nodes()} nodes, {tree.count_leaves()} leaf chunks")
    else:
        print("Cache load failed, rebuilding...")

if tree is None:
    if not DOCS_ROOT.exists() or not any(DOCS_ROOT.iterdir()):
        print("folders/ is empty. Add your documents there, then re-run this cell.")
    else:
        # If a previous run crashed, the per-node cache picks up where it left off.
        cache_status()
        tree = build_tree(DOCS_ROOT, verbose=True)
        if tree:
            save_tree(tree, TREE_FILE)

Cleared 0 cached node(s) from tree_cache/nodes
Per-node cache @ tree_cache/nodes:
  total size: 0.0 KB
Building summarisation tree from: /Users/asharma/Desktop/Project Algorithm/folders
Per-node cache: tree_cache/nodes  (delete to force a full rebuild)
  Folder: folders
  Folder: Assay_Validation
  Folder: Approved Validation Reports
  Folder: Amendments
    Parsing Amendment 1 WGS Validation Report.pdf (structure-aware) ...
      37 section(s), 541 paragraph(s)


      done: 37 section(s)  (fresh paras=541, cached=0)
  Folder: Former Assays
    Parsing Targeted Sequencing - CHARM Panel Validation Report v1.0.pdf (structure-aware) ...
      32 section(s), 399 paragraph(s)


      done: 32 section(s)  (fresh paras=399, cached=0)
  Folder: Obsolete Versions
    Parsing Plasma Whole Genome Sequencing Validation Report v1.0.pdf (structure-aware) ...
      51 section(s), 881 paragraph(s)


      done: 51 section(s)  (fresh paras=881, cached=0)
    Parsing Plasma Whole Genome Sequencing Validation Report v2.0.pdf (structure-aware) ...
      60 section(s), 1047 paragraph(s)


      done: 60 section(s)  (fresh paras=1047, cached=0)
    Parsing Targeted Sequencing - REVOLVE Panel Validation Report v1.0.pdf (structure-aware) ...
      40 section(s), 638 paragraph(s)


...

      done: 2 section(s)  (fresh paras=54, cached=0)
    Parsing HE71221_YYZL169708259_2023-01-30.pdf (structure-aware) ...
      2 section(s), 54 paragraph(s)


      done: 2 section(s)  (fresh paras=54, cached=0)
    Parsing HE71222_YYZL169708262_2023-01-30.pdf (structure-aware) ...
      2 section(s), 54 paragraph(s)


      done: 2 section(s)  (fresh paras=54, cached=0)
    Parsing 230125 WO00522325 OICR.PDF (structure-aware) ...
      5 section(s), 119 paragraph(s)


      done: 5 section(s)  (fresh paras=119, cached=0)
    Parsing 230130 WO00532623 OICR Sarah Barker.pdf (structure-aware) ...
      46 section(s), 1257 paragraph(s)


  230130 WO00532623 OICR Sarah:   1%|▉                                                                        | 17/1257 [00:27<34:11,  1.65s/it]

##### 12.5 · Repair failed summaries

If Ollama was unreachable during the build, those nodes were saved with a placeholder like `(summary unavailable: ...)`. Now that Ollama is back, `repair_tree()` re-summarises **only** those nodes — bottom-up, so a chunk is fixed before the document/folder summary that depends on it — and rewrites both the per-node cache and the saved tree. Re-runnable and safe: nodes that already have a real summary are left untouched.

In [ ]:
def _needs_summary(s: str) -> bool:
    """True if a summary is the failure placeholder (or empty)."""
    return (not s) or s.strip().startswith("(summary unavailable")


def count_unavailable(node: TreeNode) -> int:
    n = 1 if _needs_summary(node.summary) else 0
    return n + sum(count_unavailable(c) for c in node.children)


def _resummarise_chunk(node: TreeNode) -> str:
    """Recompute a single chunk/leaf summary from its stored content."""
    meta = node.metadata or {}
    src  = Path(meta.get("source_file", "")).name
    if meta.get("is_tabular"):
        hint = f"spreadsheet/CSV {src}"
    elif meta.get("page") is not None:
        hint = f"{src}, page {meta.get('page')}"
    else:
        hint = src
    return _llm_summarise(node.content or "", context_hint=hint)


def _postorder(node: TreeNode, out: List[TreeNode]) -> None:
    for c in node.children:
        _postorder(c, out)
    out.append(node)   # children always appear before their parent


def repair_tree(tree: TreeNode, refresh_parents: bool = True, verbose: bool = True) -> dict:
    """Re-summarise every node whose summary failed while Ollama was down.

    Processed bottom-up. A parent (document/folder/root) is re-combined if its
    own summary failed, or — when refresh_parents=True — if any of its children
    were just repaired (so the parent reflects the corrected child summaries).
    Any node whose subtree changed has its per-node cache file rewritten.
    """
    nodes: List[TreeNode] = []
    _postorder(tree, nodes)

    changed: set = set()    # node_ids whose summary OR subtree changed
    stats = {"chunk": 0, "document": 0, "folder": 0, "root": 0, "failed": 0}

    for node in tqdm(nodes, desc="Repairing summaries", disable=not verbose):
        child_changed = any(c.node_id in changed for c in node.children)

        if node.is_leaf():
            if _needs_summary(node.summary):
                new = _resummarise_chunk(node)
                if not _needs_summary(new):
                    node.summary = new
                    stats["chunk"] += 1
                    changed.add(node.node_id)
                else:
                    stats["failed"] += 1            # Ollama dropped again
        else:
            own_broken = _needs_summary(node.summary)
            if own_broken or (refresh_parents and child_changed):
                new = _llm_combine_summaries(
                    [c.summary for c in node.children], label=node.name
                )
                if not _needs_summary(new):
                    node.summary = new
                    stats[node.node_type] = stats.get(node.node_type, 0) + 1
                    changed.add(node.node_id)
                elif own_broken:
                    stats["failed"] += 1

        # If this node's summary changed, or any descendant did, its cache file
        # (which embeds the subtree) is stale and must be rewritten.
        if node.node_id in changed or child_changed:
            changed.add(node.node_id)
            _save_cached_node(node)

    if verbose:
        print(f"\nRepaired — chunks: {stats['chunk']}, documents: {stats['document']}, "
              f"folders: {stats['folder']}, root: {stats['root']}")
        if stats["failed"]:
            print(f"  {stats['failed']} node(s) still failing — check the Ollama "
                  f"connection and re-run this cell.")
    return stats


# --- Run the repair -------------------------------------------------------
if tree is None:
    print("No tree in memory — build or load it first (cell above).")
else:
    missing = count_unavailable(tree)
    print(f"{missing} node(s) currently have an unavailable summary.")
    if missing == 0:
        print("Nothing to repair.")
    else:
        # Make sure Ollama is reachable before we start.
        try:
            client.list()
        except Exception as e:
            print(f"Ollama still unreachable ({type(e).__name__}: {e}). "
                  f"Start the tunnel, then re-run this cell.")
        else:
            repair_tree(tree, refresh_parents=True, verbose=True)
            save_tree(tree, TREE_FILE)
            print(f"Saved repaired tree to {TREE_FILE}")
            print(f"Remaining unavailable: {count_unavailable(tree)}")

## 13 · Build the RAG baseline indices

In [ ]:
rag_index: Optional[RAGIndex] = None

if tree is not None:
    all_chunks = collect_all_chunks(tree)
    print(f"Total chunks for RAG index: {len(all_chunks)}")

    rag_index = RAGIndex(all_chunks)
    print("BM25 index built.")

    # Build vector index (comment out to skip if nomic-embed-text is unavailable)
    try:
        rag_index.build_vector_index(verbose=True)
    except Exception as e:
        print(f"Vector index skipped: {e}")
        print("BM25 retrieval will still work.")
else:
    print("No tree loaded — run cell 12 first.")

## 14 · Run queries — compare all three methods

Edit `QUERY` and run this cell to compare the tree agent against BM25 and vector search.

In [ ]:
QUERY = "What are the quality control procedures for sequencing?"

results = {}

# ── Tree Agent ──────────────────────────────────────────────────────────────
if tree:
    results["tree_agent"] = query_tree_agent(QUERY, tree, max_chunks=MAX_CHUNKS_RET, verbose=True)

# ── BM25 ────────────────────────────────────────────────────────────────────
if rag_index:
    results["bm25"] = query_rag(QUERY, rag_index, method="bm25")

# ── Vector search ───────────────────────────────────────────────────────────
if rag_index and rag_index.embeddings is not None:
    results["vector"] = query_rag(QUERY, rag_index, method="vector")

## 15 · Side-by-side comparison

In [ ]:
if results:
    print(f"\n{'='*70}")
    print(f"COMPARISON  —  Query: {QUERY}")
    print("="*70)

    rows = []
    for method, res in results.items():
        chunks_retrieved = len(res["chunks"])
        unique_docs = len({c.metadata.get("source_file") for c in res["chunks"]})
        r_time = res.get("retrieve_time") or res.get("traverse_time", 0)
        rows.append({
            "Method":          method,
            "Chunks retrieved": chunks_retrieved,
            "Unique docs":     unique_docs,
            "Retrieve time(s)": f"{r_time:.2f}",
            "Synth time(s)":   f"{res['synth_time']:.2f}",
        })

    df_compare = pd.DataFrame(rows).set_index("Method")
    display(df_compare)

    print("\n" + "─"*70)
    for method, res in results.items():
        print(f"\n[{method.upper()}] Answer:")
        print(textwrap.fill(res["answer"], width=80, initial_indent="  ", subsequent_indent="  "))
        print()

## 16 · Batch evaluation (optional)

Provide a list of `(question, expected_source_file)` pairs to measure source-retrieval precision.

In [ ]:
# Fill these in once you know your document library
EVAL_PAIRS: List[Tuple[str, str]] = [
    ("List the 9 eyewash and emergency shower stations at OICR Genomics Laboratory?", "Eyewash and Emergency Shower Review Plan.docx"),
]

if EVAL_PAIRS and tree and rag_index:
    eval_rows = []
    for question, expected_doc in tqdm(EVAL_PAIRS, desc="Evaluating"):
        for method in ["tree_agent", "bm25", "vector"]:
            if method == "tree_agent":
                res = query_tree_agent(question, tree, verbose=False)
                r_time = res["traverse_time"]
            else:
                if method == "vector" and rag_index.embeddings is None:
                    continue
                res = query_rag(question, rag_index, method=method)
                r_time = res["retrieve_time"]

            retrieved_docs = {Path(c.metadata.get("source_file", "")).name for c in res["chunks"]}
            hit = expected_doc in retrieved_docs
            eval_rows.append({
                "Question":     question[:60],
                "Expected doc": expected_doc,
                "Method":       method,
                "Hit":          hit,
                "Retrieve(s)":  f"{r_time:.2f}",
            })

    df_eval = pd.DataFrame(eval_rows)
    display(df_eval)

    print("\nPrecision by method:")
    display(df_eval.groupby("Method")["Hit"].mean().rename("Precision@k"))
else:
    print("Add (question, expected_source) pairs to EVAL_PAIRS to run evaluation.")

## 17 · Interactive query loop

In [ ]:
def interactive_query(
    use_tree:   bool = True,
    use_bm25:   bool = True,
    use_vector: bool = False,   # set True if vector index was built
):
    """Simple REPL — type 'quit' to exit."""
    if tree is None:
        print("No tree loaded — build the tree first (Section 12).")
        return
    print("Interactive SUDS Scholar — type your question, or 'quit' to stop.")
    while True:
        query = input("\nQuestion: ").strip()
        if not query or query.lower() in {"quit", "exit", "q"}:
            break
        if use_tree:
            query_tree_agent(query, tree, verbose=True)
        if use_bm25 and rag_index:
            query_rag(query, rag_index, method="bm25")
        if use_vector and rag_index and rag_index.embeddings is not None:
            query_rag(query, rag_index, method="vector")


# Uncomment to start the interactive loop:
# interactive_query(use_tree=True, use_bm25=True, use_vector=False)

## 18 · Inspect the tree structure

In [ ]:
def print_tree(node: TreeNode, indent: int = 0, max_depth: int = 4) -> None:
    """Print a visual tree overview."""
    if indent > max_depth * 2:
        return
    prefix = "  " * indent
    icon = {"root": "🗂", "folder": "📁", "document": "📄", "chunk": "📝"}.get(node.node_type, "•")
    summary_short = node.summary[:80].replace("\n", " ") + "..."
    print(f"{prefix}{icon} [{node.node_type}] {node.name}")
    print(f"{prefix}   ↳ {summary_short}")
    if indent // 2 < max_depth:
        for child in node.children:
            print_tree(child, indent + 2, max_depth)


if tree:
    print_tree(tree, max_depth=3)
else:
    print("No tree loaded.")